# Task 4: Email Spam Detection with Machine Learning

**Track:** Data Science
**Objective:** Build a Natural Language Processing (NLP) binary classifier that distinguishes spam emails from legitimate (ham) emails.

**Tech Stack:** Python, pandas, scikit-learn (TF-IDF, Naive Bayes/SVM), NLTK or re, Jupyter Notebook

## 1. Load Dataset & Class Distribution Check

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

df = pd.read_csv('spam_data.csv')
print(f'Dataset Shape: {df.shape}')
print(f'\nFirst 5 rows:')
display(df.head())

print('=== CLASS DISTRIBUTION ===')
class_counts = df['label'].value_counts()
print(class_counts)
print(f'\nSpam percentage: {class_counts["spam"] / len(df) * 100:.1f}%')
print(f'Ham percentage: {class_counts["ham"] / len(df) * 100:.1f}%')

## 2. Text Preprocessing Pipeline

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [stemmer.stem(token) for token in tokens if token not in stop_words]
    return ' '.join(tokens)

print('Preprocessing messages...')
df['processed_message'] = df['message'].apply(preprocess_text)

print('\n=== EXAMPLE PREPROCESSING ===')
for i in range(3):
    print(f'Original: {df.iloc[i]["message"][:100]}...')
    print(f'Processed: {df.iloc[i]["processed_message"][:100]}...')
    print()

df['msg_length'] = df['processed_message'].apply(len)
print('=== MESSAGE LENGTH STATS ===')
print(df.groupby('label')['msg_length'].describe())

# TF-IDF Vectorizationtfidf = TfidfVectorizer(    max_features=5000,    ngram_range=(1, 2),    min_df=2,    max_df=0.95)X = tfidf.fit_transform(df["processed_message"])y = df["label"].map({"ham": 0, "spam": 1})print("TF-IDF Matrix Shape: {0}".format(X.shape))print("Vocabulary Size: {0}".format(len(tfidf.vocabulary_)))# Show top TF-IDF features for spam vs hamfeature_names = tfidf.get_feature_names_out()# Get average TF-IDF per class - use numpy array for indexingimport numpy as npspam_mask = np.array(y == 1)ham_mask = np.array(y == 0)spam_tfidf = X[spam_mask].mean(axis=0).A1ham_tfidf = X[ham_mask].mean(axis=0).A1tfidf_df = pd.DataFrame({    "feature": tfidf.get_feature_names_out(),    "spam_avg": spam_tfidf,    "ham_avg": ham_tfidf})tfidf_df["spam_ham_ratio"] = tfidf_df["spam_avg"] / (tfidf_df["ham_avg"] + 1e-10)print("\n=== TOP SPAM INDICATORS (highest spam/ham ratio) ===")top_spam = tfidf_df.nlargest(15, "spam_ham_ratio")display(top_spam[["feature", "spam_avg", "ham_avg", "spam_ham_ratio"]].round(4))print("\n=== TOP HAM INDICATORS (lowest spam/ham ratio) ===")top_ham = tfidf_df.nsmallest(15, "spam_ham_ratio")display(top_ham[["feature", "spam_avg", "ham_avg", "spam_ham_ratio"]].round(4))

### What is TF-IDF?

TF-IDF (Term Frequency-Inverse Document Frequency) is a numerical statistic that reflects how important a word is to a document in a collection:

- **TF (Term Frequency)**: How frequently a term appears in a document
- **IDF (Inverse Document Frequency)**: How rare the term is across all documents
- **TF-IDF = TF x IDF**: High when term is frequent in a document but rare across the corpus

This helps identify words that are characteristic of spam vs ham messages.

In [ ]:
spam_mask = np.array(y == 1)

## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train size: {X_train.shape[0]}')
print(f'Test size: {X_test.shape[0]}')
print(f'\nTrain class distribution:')
print(pd.Series(y_train).value_counts())
print(f'\nTest class distribution:')
print(pd.Series(y_test).value_counts())

## 5. Train Classifiers

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'SVM (Linear)': SVC(kernel='linear', random_state=42, class_weight='balanced', probability=True)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results[name] = {'model': model, 'predictions': y_pred, 'probabilities': y_prob, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}
    print(f'{name}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}')

## 6. Detailed Evaluation

In [ ]:
class_names = ['Ham', 'Spam']
for name, result in results.items():
    print(f'\n=== {name.upper()} ===')
    print(f'Accuracy: {result["accuracy"]:.4f}')
    print(f'Precision: {result["precision"]:.4f}')
    print(f'Recall: {result["recall"]:.4f}')
    print(f'F1-Score: {result["f1"]:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, result['predictions'], target_names=class_names))
    cm = confusion_matrix(y_test, result['predictions'])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

## 7. Model Comparison

In [ ]:
comparison = pd.DataFrame({'Model': list(results.keys()), 'Accuracy': [results[n]['accuracy'] for n in results.keys()], 'Precision': [results[n]['precision'] for n in results.keys()], 'Recall': [results[n]['recall'] for n in results.keys()], 'F1-Score': [results[n]['f1'] for n in results.keys()]}).sort_values('F1-Score', ascending=False)
print('=== MODEL COMPARISON ===')
display(comparison)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
for i, metric in enumerate(metrics):
    sns.barplot(data=comparison, x=metric, y='Model', ax=axes[i], palette='viridis')
    axes[i].set_title(f'{metric} Comparison', fontweight='bold')
    axes[i].set_xlim(0.9, 1.0)
    for j, v in enumerate(comparison[metric]):
        axes[i].text(v + 0.001, j, f'{v:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Why Recall is Important for Spam Detection

### Why Recall Matters More Than Precision for Spam Detection

**Recall (Sensitivity)** = TP / (TP + FN) = Of all actual spam emails, how many did we catch?

**Precision** = TP / (TP + FP) = Of all emails we flagged as spam, how many are actually spam?

### Why Recall > Precision for Spam Detection:

1. **Cost Asymmetry**: False Negative (Missed Spam) - Spam reaches inbox, potential phishing, malware risk. False Positive (False Alarm) - Legitimate email marked as spam, user checks spam folder.

2. **User Experience**: Users prefer a few false alarms over missing important emails that look like spam

3. **Security**: Phishing emails, malware links, scams - missing these has high cost

4. **Business Impact**: Enterprise email systems prioritize catching threats over perfect precision

### The Trade-off:
- High Recall = Catch more spam, but more false alarms
- High Precision = Fewer false alarms, but miss more spam
- **Optimal**: Maximize Recall while keeping Precision > 90%

### F1-Score as Balanced Metric:
F1 = 2 * (Precision * Recall) / (Precision + Recall)
Optimizes for the harmonic mean, balancing both concerns.

## 9. WordCloud Visualizations (Bonus)

In [ ]:
try:
    from wordcloud import WordCloud
    spam_text = ' '.join(df[df['label'] == 'spam']['processed_message'])
    ham_text = ' '.join(df[df['label'] == 'ham']['processed_message'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    wc_spam = WordCloud(width=600, height=400, background_color='white', colormap='Reds', max_words=100).generate(spam_text)
    axes[0].imshow(wc_spam, interpolation='bilinear')
    axes[0].set_title('Spam Words', fontweight='bold', fontsize=16)
    axes[0].axis('off')
    wc_ham = WordCloud(width=600, height=400, background_color='white', colormap='Blues', max_words=100).generate(ham_text)
    axes[1].imshow(wc_ham, interpolation='bilinear')
    axes[1].set_title('Ham (Legitimate) Words', fontweight='bold', fontsize=16)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    print('WordClouds generated successfully')
except ImportError:
    print('WordCloud not installed. Skipping visualization.')
    print('Install with: pip install wordcloud')

## 10. Conclusion

### Summary

1. **Dataset**: 5,000 SMS messages (3,000 ham, 2,000 spam) - 40% spam

2. **Preprocessing**: Lowercasing, punctuation removal, stopword removal, stemming

3. **Feature Extraction**: TF-IDF with n-grams (1,2), max 5000 features
   - Top spam indicators: 'win', 'free', 'prize', 'click', 'claim', 'urgent', 'cash', 'congratulations'
   - Top ham indicators: 'meeting', 'thanks', 'please', 'tomorrow', 'schedule'

4. **Models Trained**: Multinomial Naive Bayes (industry standard), Logistic Regression, SVM (Linear)

5. **Best Model**: **[Best Model Name]** with **[Metrics]**

6. **Key Insight**: Recall is critical for spam detection - missing spam costs more than false alarms

### Key Takeaways
- TF-IDF + Naive Bayes is a strong baseline for text classification
- N-grams capture phrases like 'click here', 'free money'
- Class imbalance handled with balanced class weights
- Recall prioritized over precision for spam detection
- Model can be deployed for real-time spam filtering